# PC2 — Grupo 4 · Programación Concurrente y Distribuida (1ACC0065)
## Parte 1: Modelado en Promela de la lógica de sincronización

**Proyecto:** Predicción concurrente del consumo energético urbano (Regresión Lineal, dataset *Smart Meters in London*, ODS 11).

Este cuaderno cubre el primer criterio de la PC2: *planteamiento del algoritmo concurrente inicial y modelado en Promela para simular la lógica de sincronización y demostrar que esté libre de condición de carrera* (3 puntos).

El cuaderno está organizado en cinco partes: (1) Promela, (2) implementación en Go, (3) speedup con media recortada, (4) guía de análisis y (5) evidencias para el repositorio.

| Paso | Contenido | Resultado esperado |
|---|---|---|
| 1 | Planteamiento del algoritmo concurrente | Diseño Worker Pool con acumulador protegido |
| 2 | Instalación de Spin | Spin 6.5.2 |
| 3 | Modelo sin protección (`race_condition.pml`) | Spin encuentra la condición de carrera |
| 4 | Contraejemplo paso a paso | Pérdida de actualización (*lost update*) |
| 5 | Modelo corregido (`race_condition_fixed.pml`) | 0 errores, 0 estados finales inválidos |
| 6 | Evidencias y equivalencia con Go | Archivos para el repositorio y el informe |

## 1. Planteamiento del algoritmo concurrente

El modelo es una **Regresión Lineal Múltiple por Ecuaciones Normales**:

$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

Como $X^\top X$ y $X^\top y$ son **sumas por filas**, el dataset de entrenamiento (≈2.56 millones de filas hogar-día y 18 columnas con el intercepto) puede dividirse en bloques independientes:

$$X^\top X = \sum_{k=1}^{N} X_k^\top X_k \qquad\qquad X^\top y = \sum_{k=1}^{N} X_k^\top y_k$$

**Patrón elegido: Worker Pool.**

1. El productor divide las filas en *N* bloques y los envía por un canal de trabajos.
2. Cada *goroutine* worker calcula sus parciales $X_k^\top X_k$ y $X_k^\top y_k$ **en memoria local**, sin compartir nada.
3. Al terminar, el worker incorpora su parcial al **acumulador global** dentro de una **sección crítica** protegida por `sync.Mutex`.
4. `sync.WaitGroup` espera a todos los workers y luego se resuelve, de forma secuencial, el sistema 18×18.

El único punto con memoria compartida es el paso 3. Ese es el punto que se modela y verifica en Promela.

**Abstracción del modelo.** Promela no maneja matrices de números reales, así que el acumulador matricial se reduce a un entero: la suma parcial de `daily_energy_kwh` de cada bloque, que es un componente de $X^\top y$. Si la actualización de un solo componente es segura, la de toda la matriz también lo es, porque usa la misma sección crítica.

| Elemento en Go | Elemento en Promela |
|---|---|
| goroutine worker | `proctype Worker` |
| acumulador global `XtY` | `int global_sum_kwh` |
| parcial del bloque | parámetro `partial_kwh` (125 430, 98 650 y 143 210 kWh) |
| `sync.Mutex` `Lock()` / `Unlock()` | `atomic { !mutex -> mutex = true }` / `mutex = false` |
| `sync.WaitGroup` `Wait()` | `(finished == 3)` en `init` |
| verificación de resultado | `assert(global_sum_kwh == expected_sum)` |

## 2. Instalación de Spin

Colab usa Ubuntu, cuyo repositorio oficial incluye **Spin 6.5.2**, la misma versión con la que se obtuvieron las capturas del informe. También se usa `gcc`, que ya viene instalado, para compilar el verificador `pan`.

In [ ]:
!apt-get -qq install -y spin > /dev/null
!spin -V && gcc --version | head -1

## 3. Modelo sin protección: `race_condition.pml`

Tres workers suman su aporte al acumulador global en **tres pasos separados**: leer, sumar y escribir. Estos pasos no son atómicos, así que Spin puede intercalarlos de todas las formas posibles.

- Resultado correcto: 125 430 + 98 650 + 143 210 = **367 290 kWh**.
- La aserción final comprueba que el acumulador tenga ese valor cuando terminan los tres workers.

In [ ]:
%%writefile race_condition.pml
/* race_condition.pml
 * Modelo SIN protección: tres workers suman su aporte parcial (kWh)
 * a un acumulador global compartido. Se espera que Spin encuentre
 * una intercalación con pérdida de actualización (lost update).
 */
int  global_sum_kwh = 0;   /* acumulador global compartido          */
int  expected_sum   = 0;   /* suma correcta, calculada en init      */
byte finished       = 0;   /* contador de workers que terminaron    */

proctype Worker(int partial_kwh) {
    int local_sum;
    local_sum = global_sum_kwh;              /* 1. leer el acumulador   */
    local_sum = local_sum + partial_kwh;     /* 2. sumar el aporte      */
    global_sum_kwh = local_sum;              /* 3. escribir el resultado */
    finished++
}

init {
    expected_sum = 125430 + 98650 + 143210;  /* = 367290 kWh */
    run Worker(125430);
    run Worker(98650);
    run Worker(143210);
    (finished == 3);                         /* esperar a los 3 workers */
    assert(global_sum_kwh == expected_sum)
}

### 3.1 Verificación con Spin

Los tres comandos de siempre:

1. `spin -a` genera el verificador en C (`pan.c`).
2. `gcc` lo compila. La opción `-w` solo oculta advertencias del compilador que no afectan el resultado.
3. `./pan` explora el espacio de estados y guarda la salida en `resultados/` como evidencia.

In [ ]:
!mkdir -p resultados
!spin -a race_condition.pml && gcc -w -o pan pan.c && ./pan | tee resultados/race_condition_pan.txt

### 3.2 Interpretación

La salida debe coincidir con estos valores:

| Indicador | Valor | Significado |
|---|---|---|
| `assertion violated` | a profundidad 20 | Existe una ejecución que termina con una suma incorrecta |
| `errors` | 1 | Spin encontró la condición de carrera |
| `states stored / matched / transitions` | 39 / 3 / 42 | Tamaño de la búsqueda hasta hallar el error |
| `Search not completed` | — | Normal: Spin se detiene al primer error |

Spin también escribe `race_condition.pml.trail`, que contiene la ejecución exacta que produjo el error. La reproducimos en el paso siguiente.

## 4. Contraejemplo paso a paso

`spin -t -p -g -l` reproduce la traza e imprime cada paso con los valores de las variables globales (`-g`) y locales (`-l`). Si aparece el aviso *"race_condition.pml is newer than race_condition.pml.trail"*, puede ignorarse: se debe a que ambos archivos se crearon en el mismo segundo.

In [ ]:
!spin -t -p -g -l race_condition.pml | tee resultados/race_condition_trail.txt

### 4.1 Lectura de la traza

| Paso | Worker | Acción | Valor |
|---|---|---|---|
| 5–8 | Worker 3 (143 210) | lee 0, suma y escribe | `global_sum_kwh = 143 210` |
| 10–11 | Worker 2 (98 650) | lee 143 210 y suma en local | `local_sum = 241 860` |
| 12–13 | Worker 1 (125 430) | **lee el mismo 143 210** y suma en local | `local_sum = 268 640` |
| 14 | Worker 2 | escribe | `global_sum_kwh = 241 860` |
| 17 | Worker 1 | **sobrescribe** | `global_sum_kwh = 268 640` |
| 21 | init | `assert` falla | 268 640 ≠ 367 290 |

Los workers 1 y 2 leyeron el acumulador **antes** de que el otro escribiera. La segunda escritura borró a la primera y el aporte del worker 2 se perdió: 367 290 − 98 650 = 268 640. Esto es una **pérdida de actualización** (*lost update*), el caso típico de condición de carrera sobre memoria compartida.

## 5. Modelo corregido: `race_condition_fixed.pml`

Se añade un **mutex** que protege la secuencia leer-sumar-escribir:

- **Lock:** `atomic { !mutex -> mutex = true }`. El worker espera a que el mutex esté libre y lo toma en un solo paso indivisible. Sin `atomic`, dos workers podrían ver el mutex libre al mismo tiempo.
- **Sección crítica:** los tres pasos de actualización.
- **Unlock:** `mutex = false`.

El resto del modelo no cambia, para que la comparación entre ambas versiones sea directa.

In [ ]:
%%writefile race_condition_fixed.pml
/* race_condition_fixed.pml
 * Misma lógica, pero la sección crítica (leer-sumar-escribir)
 * está protegida por un mutex. Equivale a sync.Mutex en Go.
 */
int  global_sum_kwh = 0;
int  expected_sum   = 0;
byte finished       = 0;
bool mutex          = false;   /* false = libre, true = ocupado */

proctype Worker(int partial_kwh) {
    int local_sum;
    atomic { !mutex -> mutex = true };       /* Lock(): esperar y tomar  */
    local_sum = global_sum_kwh;              /* --- sección crítica ---  */
    local_sum = local_sum + partial_kwh;
    global_sum_kwh = local_sum;              /* ------------------------ */
    mutex = false;                           /* Unlock()                 */
    finished++
}

init {
    expected_sum = 125430 + 98650 + 143210;  /* = 367290 kWh */
    run Worker(125430);
    run Worker(98650);
    run Worker(143210);
    (finished == 3);
    assert(global_sum_kwh == expected_sum)
}

In [ ]:
!rm -f pan pan.* *.trail
!spin -a race_condition_fixed.pml && gcc -w -o pan pan.c && ./pan | tee resultados/race_condition_fixed_pan.txt

### 5.1 Interpretación

| Indicador | Valor | Significado |
|---|---|---|
| `errors` | **0** | Ninguna intercalación viola la aserción: no hay condición de carrera |
| `invalid end states` | + (activado), sin errores | Ningún proceso queda bloqueado: no hay *deadlock* |
| `states stored / matched / transitions` | 317 / 155 / 472 | Espacio de estados **completo** explorado |
| `depth reached` | 28 | Ejecución más larga posible |
| `unreached in proctype Worker` | 0 of 9 states | Se ejercitó todo el código del worker |
| `unreached in init` | 0 of 7 states | Se ejercitó todo el código de `init` |

A diferencia del caso anterior, aquí la búsqueda sí termina. Spin revisó **todas** las intercalaciones posibles de los tres workers y en todas el acumulador llegó a 367 290 kWh.

## 6. Evidencias y equivalencia con Go

### 6.1 Archivos para el repositorio

La celda siguiente reúne los modelos y las salidas de Spin en la carpeta `promela/`. Se descargarán junto con el resto de evidencias al final del cuaderno (Parte 5).

In [ ]:
!mkdir -p promela && cp race_condition.pml race_condition_fixed.pml promela/ && cp -r resultados promela/
!ls -R promela

### 6.2 Cómo se traslada a Go

El diseño verificado se implementará en la Parte 2 con esta estructura. Se muestra aquí solo como referencia: no es una celda ejecutable.

```go
var (
    mu  sync.Mutex
    wg  sync.WaitGroup
    XtX [P][P]float64   // acumulador global compartido
    XtY [P]float64
)

for w := 0; w < workers; w++ {
    wg.Add(1)
    go func() {
        defer wg.Done()
        for bloque := range jobs {
            pXtX, pXtY := parciales(bloque) // memoria local: sin sincronización
            mu.Lock()                       // Lock()  ≡ atomic { !mutex -> mutex = true }
            sumar(&XtX, &XtY, pXtX, pXtY)   // sección crítica ≡ leer-sumar-escribir
            mu.Unlock()                     // Unlock() ≡ mutex = false
        }
    }()
}
wg.Wait()                                   // ≡ (finished == 3)
beta := resolver(XtX, XtY)                  // secuencial, sistema 18×18
```

El mutex se toma **una sola vez por bloque**, no por cada fila. Así casi todo el cálculo ocurre en paralelo y la sección crítica es corta. Esto será relevante al analizar el speedup.

### 6.3 Checklist de capturas para el informe

1. Salida de la sección 3.1: `assertion violated`, `errors: 1`.
2. Traza de la sección 4, pasos 10 a 21.
3. Salida de la sección 5: `errors: 0`, `317 states`, `unreached 0 of 9 / 0 of 7`.

---
# Parte 2: Implementación en Go (secuencial vs. concurrente)

Esta parte cubre el criterio *implementación de código secuencial y concurrente en Go, con evidencia de funcionamiento* (4 puntos) y parte de la *explicación del algoritmo y sus mecanismos de sincronización* (3 puntos).

**Insumo:** `lcl_modelo_go.csv.gz`, generado por `model_prep.go` en la PC1 (3 285 438 filas). Se usan 2 559 364 filas de *train* para entrenar y 452 163 de *validation* para evaluar. El conjunto *test* se reserva para el TP final.

**Modelo:** 11 variables explicativas más el intercepto, es decir, **P = 12 coeficientes**. Las ecuaciones normales forman un sistema de **12×12**. La cifra de "18 features" que aparece en el informe corresponde a las 18 columnas del CSV, no a las variables del modelo.

| Variable | Descripción |
|---|---|
| `tariff_tou`, `is_weekend` | Tarifa ToU y fin de semana (0/1) |
| `dow_sin`, `dow_cos`, `month_sin`, `month_cos` | Calendario en codificación cíclica |
| `day_index` | Tendencia temporal (días desde 2011-01-01) |
| `lag_1_kwh`, `lag_7_kwh` | Consumo del día anterior y de hace 7 días |
| `rolling_mean_7_kwh`, `rolling_std_7_kwh` | Media y desviación de los 7 días previos |
| **Objetivo:** `target_daily_kwh_capped` | Consumo diario winsorizado (kWh) |

## 2.1 Entorno de ejecución

Los resultados de rendimiento dependen del hardware, así que el informe debe indicar dónde se midieron. Colab gratuito suele asignar **2 vCPU**, por lo que aquí el speedup máximo posible es cercano a 2.

In [ ]:
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread\(s\) per core|Core\(s\) per socket'
!free -h | head -2

## 2.2 Instalación de Go

Se instala la versión oficial de Go, porque la del repositorio de Ubuntu es antigua. Solo se usa la **biblioteca estándar**, como exige el enunciado.

In [ ]:
!wget -q https://go.dev/dl/go1.22.12.linux-amd64.tar.gz -O /tmp/go.tgz && rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz
import os
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
!go version

## 2.3 Dataset

Primero se busca el archivo en la carpeta de Drive del notebook de la PC1. Si no está ahí, se pide subirlo. Luego se copia al disco local de Colab, porque leer 700 MB directamente desde Drive es lento.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DATA = Path('/content/drive/MyDrive/Grupo4_PCD_GO/lcl_modelo_go.csv.gz')
if not DATA.exists():
    from google.colab import files
    print('No se encontró en Drive. Suba lcl_modelo_go.csv.gz:')
    DATA = Path(next(iter(files.upload())))

!cp "{DATA}" /content/lcl_modelo_go.csv.gz
DATA = '/content/lcl_modelo_go.csv.gz'
!ls -lh {DATA}

## 2.4 Diseño del programa `regresion.go`

Un solo programa con tres modos (`-mode secuencial | concurrente | benchmark`):

| Función | Responsabilidad |
|---|---|
| `loadData` | Lee el CSV (o `.gz`) y guarda X en **un solo slice contiguo** fila por fila, para aprovechar la caché |
| `accumulate` | **Núcleo común:** suma $X_k^\top X_k$ y $X_k^\top y_k$ de un rango de filas. Solo calcula el triángulo superior, porque la matriz es simétrica |
| `normalEqSequential` | Llama a `accumulate` una vez sobre todas las filas |
| `normalEqConcurrent` | **Worker Pool:** canal `jobs`, W goroutines, acumuladores locales, `sync.Mutex` y `sync.WaitGroup` |
| `solve` | Eliminación gaussiana con pivoteo parcial (12×12, secuencial) |
| `evaluate` | RMSE y R² en train y validation |
| `measure` | Tiempo real, **tiempo de CPU** (`getrusage`), memoria asignada y ciclos de GC |

**Decisión clave.** La versión secuencial y la concurrente usan **la misma función `accumulate`**. Así, la diferencia de tiempo se debe solo a la concurrencia y no a que las implementaciones sean distintas. Por eso, con 1 worker el tiempo debe ser prácticamente igual al secuencial.

### Worker Pool y sincronización

```
               ┌──────────── canal jobs (bloques de 16 384 filas) ────────────┐
 productor ──► │ [0,16384) [16384,32768) [32768,49152) ...                    │
               └──────┬──────────────┬──────────────┬─────────────────────────┘
                      ▼              ▼              ▼
                  worker 1       worker 2   ...  worker W     ← toman bloques según se liberan
               XtX,Xty locales  (sin memoria compartida)
                      │              │              │
                      └──── mu.Lock() → suma al acumulador global → mu.Unlock() ────┘
                                         (1 vez por worker)
                                              ▼
                                   wg.Wait() → solve(12×12)
```

| Mecanismo | Uso | Equivalente en Promela |
|---|---|---|
| `chan block` | Reparto **dinámico** de bloques: un worker libre toma el siguiente | — |
| memoria local por worker | Cálculo sin contención | `local_sum` |
| `sync.Mutex` | Sección crítica al fusionar en el acumulador global | `atomic { !mutex -> mutex = true }` … `mutex = false` |
| `sync.WaitGroup` | Esperar a todos antes de resolver | `(finished == 3)` |

El mutex se toma **W veces en total**, no una vez por fila ni por bloque. Con 2.5 millones de filas, la sección crítica ocupa una fracción despreciable del tiempo.

In [ ]:
!mkdir -p regresion resultados

In [ ]:
%%writefile regresion/go.mod
module regresion

go 1.22

In [ ]:
%%writefile regresion/regresion.go
// regresion.go — Regresión Lineal Múltiple por Ecuaciones Normales.
// Versión secuencial y concurrente (Worker Pool + sync.Mutex + sync.WaitGroup).
// Solo biblioteca estándar de Go.
package main

import (
	"bufio"
	"compress/gzip"
	"encoding/csv"
	"flag"
	"fmt"
	"io"
	"math"
	"os"
	"runtime"
	"sort"
	"strconv"
	"strings"
	"sync"
	"syscall"
	"time"
)

// Variables explicativas (11) + intercepto = P = 12 coeficientes.
var featureCols = []string{
	"tariff_tou", "is_weekend", "dow_sin", "dow_cos", "month_sin", "month_cos",
	"day_index", "lag_1_kwh", "lag_7_kwh", "rolling_mean_7_kwh", "rolling_std_7_kwh",
}

const targetCol = "target_daily_kwh_capped"

var P = len(featureCols) + 1

// Dataset guarda X en un solo slice contiguo (fila por fila) para aprovechar la caché.
type Dataset struct {
	N int
	X []float64 // N*P, la columna 0 es el intercepto (=1)
	Y []float64
}

func (d *Dataset) add(row []float64, y float64) {
	d.X = append(d.X, row...)
	d.Y = append(d.Y, y)
	d.N++
}

// ---------------------------------------------------------------- carga

func loadData(path string, limit int) (train, valid Dataset, err error) {
	f, err := os.Open(path)
	if err != nil {
		return
	}
	defer f.Close()
	var r io.Reader = bufio.NewReaderSize(f, 4<<20)
	if strings.HasSuffix(path, ".gz") {
		gz, e := gzip.NewReader(r)
		if e != nil {
			err = e
			return
		}
		defer gz.Close()
		r = gz
	}
	cr := csv.NewReader(bufio.NewReaderSize(r, 4<<20))
	cr.ReuseRecord = true
	header, err := cr.Read()
	if err != nil {
		return
	}
	idx := map[string]int{}
	for i, h := range header {
		idx[strings.TrimSpace(strings.TrimPrefix(h, "\xef\xbb\xbf"))] = i
	}
	cols := make([]int, len(featureCols))
	for i, c := range featureCols {
		j, ok := idx[c]
		if !ok {
			err = fmt.Errorf("falta columna %s", c)
			return
		}
		cols[i] = j
	}
	yi, si := idx[targetCol], idx["dataset_split"]
	row := make([]float64, P)
	for {
		rec, e := cr.Read()
		if e == io.EOF {
			break
		}
		if e != nil {
			err = e
			return
		}
		split := rec[si]
		if split == "test" {
			continue // el conjunto de prueba se reserva para el TP final
		}
		row[0] = 1
		ok := true
		for i, j := range cols {
			v, e := strconv.ParseFloat(rec[j], 64)
			if e != nil || math.IsNaN(v) {
				ok = false
				break
			}
			row[i+1] = v
		}
		y, e := strconv.ParseFloat(rec[yi], 64)
		if !ok || e != nil {
			continue
		}
		if split == "train" {
			train.add(row, y)
			if limit > 0 && train.N >= limit {
				break
			}
		} else {
			valid.add(row, y)
		}
	}
	return
}

// ---------------------------------------------------------------- núcleo

// accumulate suma X_k^T X_k y X_k^T y_k para las filas [lo, hi) en xtx y xty.
// Es la MISMA función para la versión secuencial y la concurrente, así la
// comparación mide solo el efecto de la concurrencia. Solo se calcula el
// triángulo superior de XtX (es simétrica) y se completa al final.
func accumulate(d *Dataset, lo, hi int, xtx, xty []float64) {
	for r := lo; r < hi; r++ {
		x := d.X[r*P : r*P+P]
		y := d.Y[r]
		for i := 0; i < P; i++ {
			xi := x[i]
			xty[i] += xi * y
			rowI := xtx[i*P : i*P+P]
			for j := i; j < P; j++ {
				rowI[j] += xi * x[j]
			}
		}
	}
}

func mirror(xtx []float64) {
	for i := 0; i < P; i++ {
		for j := 0; j < i; j++ {
			xtx[i*P+j] = xtx[j*P+i]
		}
	}
}

// Secuencial: un solo recorrido sobre todas las filas.
func normalEqSequential(d *Dataset) (xtx, xty []float64) {
	xtx, xty = make([]float64, P*P), make([]float64, P)
	accumulate(d, 0, d.N, xtx, xty)
	mirror(xtx)
	return
}

type block struct{ lo, hi int }

// Concurrente: Worker Pool.
//   - El productor envía bloques de filas por el canal jobs.
//   - Cada worker acumula en memoria LOCAL todos los bloques que recibe.
//   - Al terminar, fusiona su parcial en el acumulador global dentro de la
//     sección crítica (sync.Mutex) — mismo patrón verificado en Promela.
//   - sync.WaitGroup espera a todos los workers antes de resolver.
func normalEqConcurrent(d *Dataset, workers, blockSize int) (xtx, xty []float64) {
	xtx, xty = make([]float64, P*P), make([]float64, P)
	var mu sync.Mutex
	var wg sync.WaitGroup
	jobs := make(chan block, workers*2)

	for w := 0; w < workers; w++ {
		wg.Add(1)
		go func() {
			defer wg.Done()
			lxtx, lxty := make([]float64, P*P), make([]float64, P) // memoria local
			for b := range jobs {
				accumulate(d, b.lo, b.hi, lxtx, lxty)
			}
			mu.Lock() // ---- sección crítica: una vez por worker ----
			for i := range lxtx {
				xtx[i] += lxtx[i]
			}
			for i := range lxty {
				xty[i] += lxty[i]
			}
			mu.Unlock()
		}()
	}
	for lo := 0; lo < d.N; lo += blockSize {
		hi := lo + blockSize
		if hi > d.N {
			hi = d.N
		}
		jobs <- block{lo, hi}
	}
	close(jobs)
	wg.Wait()
	mirror(xtx)
	return
}

// ---------------------------------------------------------------- solución y métricas

// solve resuelve A·beta = b por eliminación gaussiana con pivoteo parcial (P×P).
func solve(A, b []float64) ([]float64, error) {
	n := len(b)
	M := make([]float64, n*(n+1))
	for i := 0; i < n; i++ {
		copy(M[i*(n+1):], A[i*n:i*n+n])
		M[i*(n+1)+n] = b[i]
	}
	for c := 0; c < n; c++ {
		p := c
		for r := c + 1; r < n; r++ {
			if math.Abs(M[r*(n+1)+c]) > math.Abs(M[p*(n+1)+c]) {
				p = r
			}
		}
		if math.Abs(M[p*(n+1)+c]) < 1e-12 {
			return nil, fmt.Errorf("matriz singular en columna %d", c)
		}
		for k := 0; k <= n; k++ {
			M[c*(n+1)+k], M[p*(n+1)+k] = M[p*(n+1)+k], M[c*(n+1)+k]
		}
		for r := c + 1; r < n; r++ {
			f := M[r*(n+1)+c] / M[c*(n+1)+c]
			for k := c; k <= n; k++ {
				M[r*(n+1)+k] -= f * M[c*(n+1)+k]
			}
		}
	}
	beta := make([]float64, n)
	for i := n - 1; i >= 0; i-- {
		s := M[i*(n+1)+n]
		for k := i + 1; k < n; k++ {
			s -= M[i*(n+1)+k] * beta[k]
		}
		beta[i] = s / M[i*(n+1)+i]
	}
	return beta, nil
}

func evaluate(d *Dataset, beta []float64) (rmse, r2 float64) {
	var mean float64
	for _, y := range d.Y {
		mean += y
	}
	mean /= float64(d.N)
	var sse, sst float64
	for r := 0; r < d.N; r++ {
		x := d.X[r*P : r*P+P]
		var pred float64
		for i := 0; i < P; i++ {
			pred += beta[i] * x[i]
		}
		e := d.Y[r] - pred
		sse += e * e
		sst += (d.Y[r] - mean) * (d.Y[r] - mean)
	}
	return math.Sqrt(sse / float64(d.N)), 1 - sse/sst
}

// ---------------------------------------------------------------- medición

func cpuTime() time.Duration { // tiempo de CPU usuario+sistema del proceso
	var ru syscall.Rusage
	syscall.Getrusage(syscall.RUSAGE_SELF, &ru)
	return time.Duration(ru.Utime.Nano() + ru.Stime.Nano())
}

type run struct {
	mode    string
	workers int
	rep     int
	wall    time.Duration
	cpu     time.Duration
	alloc   uint64
	gc      uint32
}

func measure(mode string, workers, blockSize int, d *Dataset) run {
	var m0, m1 runtime.MemStats
	runtime.GC()
	runtime.ReadMemStats(&m0)
	c0 := cpuTime()
	t0 := time.Now()
	if mode == "secuencial" {
		normalEqSequential(d)
	} else {
		normalEqConcurrent(d, workers, blockSize)
	}
	wall := time.Since(t0)
	cpu := cpuTime() - c0
	runtime.ReadMemStats(&m1)
	return run{mode, workers, 0, wall, cpu, m1.TotalAlloc - m0.TotalAlloc, m1.NumGC - m0.NumGC}
}

func trimmedMean(xs []float64, frac float64) float64 {
	s := append([]float64(nil), xs...)
	sort.Float64s(s)
	k := int(float64(len(s)) * frac)
	s = s[k : len(s)-k]
	var sum float64
	for _, v := range s {
		sum += v
	}
	return sum / float64(len(s))
}

func parseInts(s string) []int {
	var out []int
	for _, p := range strings.Split(s, ",") {
		if v, err := strconv.Atoi(strings.TrimSpace(p)); err == nil && v > 0 {
			out = append(out, v)
		}
	}
	return out
}

func ms(d time.Duration) float64 { return float64(d.Nanoseconds()) / 1e6 }

// ---------------------------------------------------------------- main

func main() {
	input := flag.String("input", "lcl_modelo_go.csv.gz", "dataset de modelo (.csv o .csv.gz)")
	mode := flag.String("mode", "secuencial", "secuencial | concurrente | benchmark")
	workers := flag.Int("workers", runtime.NumCPU(), "número de workers (modo concurrente)")
	blockSize := flag.Int("block", 16384, "filas por bloque enviado al Worker Pool")
	reps := flag.Int("reps", 20, "repeticiones por configuración (benchmark)")
	wlist := flag.String("wlist", "1,2,4,8,16,32", "lista de workers (benchmark)")
	out := flag.String("out", "benchmark_runs.csv", "CSV con cada medición (benchmark)")
	limit := flag.Int("limit", 0, "máximo de filas de train (0 = todas)")
	flag.Parse()

	fmt.Printf("CPU lógicas: %d | GOMAXPROCS: %d | Go %s\n", runtime.NumCPU(), runtime.GOMAXPROCS(0), runtime.Version())
	t0 := time.Now()
	train, valid, err := loadData(*input, *limit)
	if err != nil {
		fmt.Fprintln(os.Stderr, "ERROR:", err)
		os.Exit(1)
	}
	fmt.Printf("Train: %d filas | Validation: %d filas | P = %d | carga: %s\n",
		train.N, valid.N, P, time.Since(t0).Round(time.Millisecond))

	switch *mode {
	case "secuencial", "concurrente":
		var xtx, xty []float64
		t := time.Now()
		if *mode == "secuencial" {
			xtx, xty = normalEqSequential(&train)
		} else {
			xtx, xty = normalEqConcurrent(&train, *workers, *blockSize)
		}
		acc := time.Since(t)
		beta, err := solve(xtx, xty)
		if err != nil {
			fmt.Fprintln(os.Stderr, "ERROR:", err)
			os.Exit(1)
		}
		fmt.Printf("\n=== %s", strings.ToUpper(*mode))
		if *mode == "concurrente" {
			fmt.Printf(" (workers=%d, bloque=%d)", *workers, *blockSize)
		}
		fmt.Printf(" ===\nTiempo XtX/Xty: %.2f ms\n", ms(acc))
		fmt.Println("Coeficientes (intercepto primero):")
		names := append([]string{"intercepto"}, featureCols...)
		for i, b := range beta {
			fmt.Printf("  %-20s % .10f\n", names[i], b)
		}
		rt, r2t := evaluate(&train, beta)
		rv, r2v := evaluate(&valid, beta)
		fmt.Printf("Train      -> RMSE %.4f | R2 %.4f\n", rt, r2t)
		fmt.Printf("Validation -> RMSE %.4f | R2 %.4f\n", rv, r2v)

	case "benchmark":
		f, err := os.Create(*out)
		if err != nil {
			fmt.Fprintln(os.Stderr, "ERROR:", err)
			os.Exit(1)
		}
		w := csv.NewWriter(f)
		w.Write([]string{"modo", "workers", "rep", "wall_ms", "cpu_ms", "alloc_kb", "gc"})
		configs := []int{0}
		configs = append(configs, parseInts(*wlist)...)
		fmt.Println()
		for _, nw := range configs {
			m := "concurrente"
			if nw == 0 {
				m = "secuencial"
			}
			measure(m, nw, *blockSize, &train) // calentamiento, no se registra
			walls := make([]float64, 0, *reps)
			for r := 1; r <= *reps; r++ {
				res := measure(m, nw, *blockSize, &train)
				res.rep = r
				walls = append(walls, ms(res.wall))
				w.Write([]string{res.mode, strconv.Itoa(nw), strconv.Itoa(r),
					strconv.FormatFloat(ms(res.wall), 'f', 4, 64),
					strconv.FormatFloat(ms(res.cpu), 'f', 4, 64),
					strconv.FormatFloat(float64(res.alloc)/1024, 'f', 2, 64),
					strconv.Itoa(int(res.gc))})
			}
			w.Flush()
			fmt.Printf("%-11s workers=%-3d media recortada 10%% = %8.2f ms\n", m, nw, trimmedMean(walls, 0.10))
		}
		f.Close()
		fmt.Println("Mediciones guardadas en", *out)
	default:
		fmt.Fprintln(os.Stderr, "modo no reconocido:", *mode)
		os.Exit(2)
	}
}

## 2.5 Compilación y análisis estático

`gofmt` revisa el formato y `go vet` detecta errores comunes. Ambos deben terminar sin mensajes.

In [ ]:
!cd regresion && gofmt -l . && go vet . && go build -o regresion . && echo "Compilación correcta"

## 2.6 Evidencia: ejecución secuencial

In [ ]:
!./regresion/regresion -input {DATA} -mode secuencial | tee resultados/ejecucion_secuencial.txt

## 2.7 Evidencia: ejecución concurrente

In [ ]:
!./regresion/regresion -input {DATA} -mode concurrente -workers 4 | tee resultados/ejecucion_concurrente.txt

### 2.7.1 ¿La concurrencia altera el modelo?

Se comparan los 12 coeficientes de ambas ejecuciones, impresos con 10 decimales. Deben coincidir o diferir en menos de $10^{-10}$. Cualquier diferencia mínima se debe al **orden de las sumas en punto flotante**, que no es asociativo, y no a un error de sincronización. Una diferencia grande indicaría una condición de carrera.

In [ ]:
import re
def coef(path):
    return {m[0]: float(m[1]) for m in re.findall(r'^\s{2}(\w+)\s+(\S+)$', open(path).read(), re.M)}
s, c = coef('resultados/ejecucion_secuencial.txt'), coef('resultados/ejecucion_concurrente.txt')
dif = {k: abs(s[k] - c[k]) for k in s}
print(f'{len(dif)} coeficientes comparados | diferencia máxima absoluta: {max(dif.values()):.2e}')

## 2.8 Detector de condiciones de carrera de Go

Promela verificó el **modelo** del protocolo. `go run -race` verifica el **código real**: instrumenta cada acceso a memoria y reporta `WARNING: DATA RACE` si dos goroutines acceden a la misma variable sin sincronización. Se usan 200 000 filas porque el detector hace la ejecución unas 10 veces más lenta.

El resultado esperado es que **no aparezca ningún `DATA RACE`**. Para comprobar que el detector funciona, puede comentar temporalmente `mu.Lock()` y `mu.Unlock()` y volver a ejecutar: la advertencia debe aparecer.

In [ ]:
!cd regresion && go run -race . -input {DATA} -mode concurrente -workers 8 -limit 200000 2>&1 | tee ../resultados/race_detector.txt | tail -4
!grep -q "DATA RACE" resultados/race_detector.txt && echo "ATENCIÓN: se detectaron condiciones de carrera" || echo "Resultado: 0 condiciones de carrera detectadas"

---
# Parte 3: Speedup y media recortada

Esta parte cubre el criterio *cálculo de Speedup (T-Secuencial / T-Concurrente) y media recortada tras múltiples ejecuciones, con tabla y estadística* (3 puntos).

**Protocolo de medición:**

1. Solo se mide la construcción de $X^\top X$ y $X^\top y$, que es la parte paralelizable. No se incluyen la carga del CSV ni la resolución 12×12.
2. Cada configuración (secuencial y 1, 2, 4, 8, 16 y 32 workers) tiene **1 ejecución de calentamiento** que no se registra y luego **20 repeticiones**.
3. Antes de cada repetición se llama a `runtime.GC()`, para que el recolector de basura no interfiera.
4. **Media recortada al 10 %:** se ordenan las 20 mediciones, se descartan las 2 más bajas y las 2 más altas, y se promedian las 16 restantes.
5. Se registran también la desviación estándar, el coeficiente de variación (CV) y el tiempo de CPU. Así la tabla incluye estadística y no solo promedios.

In [ ]:
!./regresion/regresion -input {DATA} -mode benchmark -reps 20 -wlist 1,2,4,8,16,32 -out resultados/benchmark_runs.csv

## 3.1 Tabla estadística

| Columna | Fórmula / significado |
|---|---|
| `media_recortada_ms` | Media de las 16 mediciones centrales |
| `desv_est_ms`, `CV_%` | Dispersión. Un CV menor al 10 % indica mediciones estables |
| `speedup` | $S = T_{sec} / T_{conc}$ |
| `eficiencia` | $E = S / W$ |
| `nucleos_usados` | Tiempo de CPU / tiempo real: cuántos núcleos trabajaron en promedio |
| `karp_flatt` | Fracción serial experimental $e = \dfrac{1/S - 1/p}{1 - 1/p}$, con $p$ = núcleos efectivos |

**Para analizar una medición hecha en otra PC**, por ejemplo en la de 8 núcleos, ejecute allí `go run . -mode benchmark -input <ruta> -out benchmark_runs.csv`. Luego suba ese CSV a `resultados/` y cambie `NUCLEOS` en la celda siguiente.

In [ ]:
import pandas as pd, numpy as np, os
from scipy import stats

runs = pd.read_csv('resultados/benchmark_runs.csv')
# Si analiza un CSV medido en otra máquina (p. ej., la PC de 8 núcleos), ajuste este valor.
NUCLEOS = os.cpu_count()

def resumen(g):
    w = g['wall_ms']
    return pd.Series({
        'n': len(w),
        'media_recortada_ms': stats.trim_mean(w, 0.10),   # descarta 10 % arriba y 10 % abajo
        'mediana_ms': w.median(),
        'desv_est_ms': w.std(ddof=1),
        'CV_%': 100 * w.std(ddof=1) / w.mean(),
        'min_ms': w.min(),
        'max_ms': w.max(),
        'cpu_ms': stats.trim_mean(g['cpu_ms'], 0.10),
        'alloc_kb': g['alloc_kb'].mean(),
        'gc': g['gc'].sum(),
    })

tabla = pd.DataFrame([{'modo': m, 'workers': k, **resumen(g)} for (m, k), g in runs.groupby(['modo', 'workers'])])
tabla = tabla.sort_values('workers').reset_index(drop=True)
t_seq = tabla.loc[tabla.modo == 'secuencial', 'media_recortada_ms'].iloc[0]

tabla['speedup'] = t_seq / tabla['media_recortada_ms']
tabla['eficiencia'] = np.where(tabla.workers > 0, tabla['speedup'] / tabla['workers'].clip(lower=1), 1.0)
tabla['nucleos_usados'] = tabla['cpu_ms'] / tabla['media_recortada_ms']     # CPU / tiempo real
# Karp–Flatt: fracción serial experimental con p = núcleos realmente disponibles
p = np.minimum(tabla['workers'], NUCLEOS)
tabla['karp_flatt'] = np.where(p > 1, (1 / tabla['speedup'] - 1 / p) / (1 - 1 / p), np.nan)

cols = ['modo', 'workers', 'n', 'media_recortada_ms', 'desv_est_ms', 'CV_%', 'min_ms', 'max_ms',
        'speedup', 'eficiencia', 'nucleos_usados', 'karp_flatt', 'alloc_kb', 'gc']
tabla[cols].to_csv('resultados/tabla_speedup.csv', index=False)
print(f'Núcleos lógicos del entorno: {NUCLEOS}')
tabla[cols].round(3)

In [ ]:
import matplotlib.pyplot as plt

AZUL, GRIS, TINTA, TINTA2 = '#2a78d6', '#8f8e89', '#0b0b0b', '#52514e'
plt.rcParams.update({'font.size': 10, 'axes.edgecolor': '#c9c8c2', 'axes.labelcolor': TINTA2,
                     'xtick.color': TINTA2, 'ytick.color': TINTA2, 'axes.spines.top': False,
                     'axes.spines.right': False})
conc = tabla[tabla.modo == 'concurrente']
w = conc['workers'].to_numpy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(w, np.minimum(w, NUCLEOS), '--', color=GRIS, lw=1.2, label=f'Ideal limitado a {NUCLEOS} núcleos')
ax.axhline(1, color=GRIS, lw=0.8, ls=':')
ax.plot(w, conc['speedup'], '-o', color=AZUL, lw=2, ms=7, label='Speedup medido')
for x, s in zip(w, conc['speedup']):
    ax.annotate(f'{s:.2f}x', (x, s), textcoords='offset points', xytext=(0, 8), ha='center', color=TINTA, fontsize=9)
ax.set_ylim(0, max(conc['speedup'].max(), NUCLEOS) * 1.18)
ax.set_xscale('log', base=2); ax.set_xticks(w); ax.set_xticklabels(w)
ax.set_xlabel('Workers (goroutines)'); ax.set_ylabel('Speedup = T secuencial / T concurrente')
ax.set_title('Speedup vs. número de workers (media recortada 10 %)', color=TINTA, loc='left')
ax.grid(axis='y', color='#ecebe6'); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig('resultados/speedup.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.bar([str(x) for x in w], conc['eficiencia'], color=AZUL, width=0.55)
for i, e in enumerate(conc['eficiencia']):
    ax.annotate(f'{e:.2f}', (i, e), textcoords='offset points', xytext=(0, 4), ha='center', color=TINTA, fontsize=9)
ax.set_xlabel('Workers (goroutines)'); ax.set_ylabel('Eficiencia = Speedup / workers'); ax.set_ylim(0, 1.2)
ax.set_title('Eficiencia paralela', color=TINTA, loc='left'); ax.grid(axis='y', color='#ecebe6')
fig.tight_layout(); fig.savefig('resultados/eficiencia.png', dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
orden = tabla.assign(etq=lambda d: np.where(d.workers == 0, 'sec', d.workers.astype(str)))
datos = [runs[(runs.modo == r.modo) & (runs.workers == r.workers)]['wall_ms'] for r in orden.itertuples()]
ax.boxplot(datos, patch_artist=True,
           boxprops=dict(facecolor='#cde2fb', color=AZUL), medianprops=dict(color=AZUL, lw=2),
           whiskerprops=dict(color=GRIS), capprops=dict(color=GRIS), flierprops=dict(markeredgecolor=GRIS))
ax.set_xticks(range(1, len(datos) + 1)); ax.set_xticklabels(orden['etq'])
ax.set_xlabel('Configuración (sec = secuencial; número = workers)'); ax.set_ylabel('Tiempo XᵀX / Xᵀy (ms)')
ax.set_title('Dispersión de las 20 mediciones por configuración', color=TINTA, loc='left')
ax.grid(axis='y', color='#ecebe6'); fig.tight_layout(); fig.savefig('resultados/dispersion.png', dpi=200); plt.show()

---
# Parte 4: Guía de análisis (speedup, escalabilidad, trade-offs y recursos)

Esta parte apoya los criterios de *análisis de speedup, escalabilidad y trade-offs* (3 puntos) y de *uso de recursos y punto de equilibrio* (2 puntos). Las preguntas siguientes deben responderse en el informe **con los números de la tabla anterior**.

**1. ¿Qué pasa con 1 worker?** Como ambas versiones comparten `accumulate`, el speedup con 1 worker debe estar cerca de **1.0x**. La diferencia es el costo real de crear una goroutine, pasar los bloques por el canal y tomar el mutex una vez, que es de microsegundos. Si en otra implementación 1 worker tarda mucho más que el secuencial (por ejemplo, 0.59x), la causa no es la concurrencia sino una diferencia en el código. El docente lo notará.

**2. ¿Hasta dónde escala?** El speedup crece casi linealmente hasta igualar el número de **núcleos físicos**. Los hilos lógicos (Hyperthreading) comparten unidades de ejecución y aportan menos. Pasado ese punto, la curva se aplana: más goroutines que núcleos solo añaden planificación.

**3. Ley de Amdahl y Karp–Flatt.** La parte secuencial dentro del tiempo medido es mínima: W fusiones de 156 números y el cierre del canal. Por eso la métrica de Karp–Flatt debería salir cercana a 0. Si crece con W, el cuello de botella es otro, por ejemplo el **ancho de banda de memoria**: cada fila son 96 bytes y se recorren unos 245 MB por ejecución, así que con muchos núcleos la CPU espera a la RAM.

**4. Trade-offs.**

| Aspecto | Secuencial | Concurrente |
|---|---|---|
| Tiempo | Base | ≈ T/núcleos hasta saturar |
| Complejidad del código | Mínima | Canal, mutex, WaitGroup y verificación (Promela, `-race`) |
| Memoria extra | 0 | W × (12×12 + 12) floats ≈ 1.2 KB por worker, despreciable |
| Reproducibilidad numérica | Exacta | Diferencias de ~$10^{-12}$ por el orden de las sumas |
| Riesgo | Ninguno | Condición de carrera si se omite el mutex (demostrado en Promela) |

**5. Uso de recursos y punto de equilibrio.**
- `nucleos_usados` cercano a W (hasta el total de núcleos) indica que el paralelismo es real.
- `alloc_kb` crece con W solo por los acumuladores locales.
- Si `gc = 0`, el recolector de basura no interviene.
- El **punto de equilibrio** es la menor cantidad de workers a partir de la cual el speedup deja de crecer de forma apreciable (por ejemplo, menos de 5 % de mejora) mientras la eficiencia sigue cayendo. Normalmente coincide con el número de núcleos físicos.

**6. Comparación entre entornos.** Si se incluyen la PC de 8 núcleos y Colab (2 vCPU), el mismo código muestra que el techo de speedup lo fija el hardware disponible. Es un buen argumento de **escalabilidad**.

---
# Parte 5: Evidencias para el repositorio

La celda siguiente empaqueta todo lo que debe subirse a GitHub:
- `promela/`: los modelos `.pml` y las salidas de Spin.
- `regresion/`: `go.mod` y `regresion.go`.
- `resultados/`: las ejecuciones, el detector de carreras, las mediciones del benchmark, la tabla y los gráficos.

El dataset **no** se sube, porque ya está en `.gitignore`.

In [ ]:
!mkdir -p entrega && cp -r promela regresion resultados entrega/ && rm -f entrega/regresion/regresion
!cd entrega && zip -qr ../evidencias_pc2.zip . && cd .. && unzip -l evidencias_pc2.zip | tail -n +4 | head -30
from google.colab import files
files.download('evidencias_pc2.zip')